In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/sushantbhardwaj15@gmail.com/FMCG_Analytics/setup_utils_config/Utilities

In [0]:
silver_orders = spark.table(f"{catalog}.{silver_schema}.orders")
for table_name in ["orders"]:
    if spark.catalog.tableExists(f"{catalog}.{gold_schema}.sb_fact_{table_name}"):
        watermark = (
            spark.table(f"{catalog}.{gold_schema}.sb_fact_{table_name}")
            .agg(F.max("_ingested_at"))
            .collect()[0][0]
        )                                                       
        new_rows = silver_orders.filter(F.col("_ingested_at") > watermark)
        print(f" Watermark: {watermark} — {new_rows.count()} new rows")
    else:
        new_rows = silver_orders
        print(f" First run record count— {new_rows.count()} rows")

new_rows = (
    new_rows
    .withColumn("_gold_loaded_at", F.current_timestamp())
    .withColumnRenamed("customer_id", "customer_code")
    .withColumnRenamed("order_qty", "sold_quantity")
    .withColumnRenamed("order_placement_date", "date")
)

In [0]:
gold_table = f"{catalog}.{gold_schema}.sb_fact_orders"

if not spark.catalog.tableExists(gold_table):
    print("Creating new table")

    (
        new_rows.write
        .format("delta")
        .option("delta.enableChangeDataFeed", "true")
        .option("mergeSchema", "true")
        .mode("overwrite")
        .saveAsTable(gold_table)
    )

else:
    gold_delta = DeltaTable.forName(spark, gold_table)

    (
        gold_delta.alias("source")
        .merge(
            new_rows.alias("gold"),
            """
            source.date = gold.date
            AND source.order_id = gold.order_id
            AND source.product_code = gold.product_code
            AND source.customer_code = gold.customer_code
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
write_audit(
    "gold",
    "orders",
    new_rows.count(),
    "SUCCESS",
)

In [0]:
display(new_rows)

## Merging with Parent company

In [0]:
df_monthly = (
    new_rows
    # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn("month_start", F.trunc("date", "MM"))   # or F.date_trunc("month", "date").cast("date")

    # 2.Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )

    # 3. Rename month_start back to `date` to match your target schema
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(20, truncate=False)

In [0]:
df_monthly.select('date').distinct().orderBy('date').show()

In [0]:
# df_monthly_recalc = (
#     monthly_table
#     .withColumn("month_start", F.trunc("date", "MM"))
#     .groupBy("month_start", "product_code", "customer_code")
#     .agg(F.sum("sold_quantity").alias("sold_quantity"))
#     .withColumnRenamed("month_start", "date")   # month_start → date = first of month
# )

# df_monthly_recalc.show(10, truncate=False)

In [0]:
df_monthly.count()

In [0]:
gold_parent_delta = DeltaTable.forName(
    spark,
    f"{catalog}.{gold_schema}.fact_orders"
)

(
    gold_parent_delta.alias("parent_gold")
    .merge(
        df_monthly.alias("child_gold"),
        """
        parent_gold.date = child_gold.date
        AND parent_gold.product_code = child_gold.product_code
        AND parent_gold.customer_code = child_gold.customer_code
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)